# MediCare Patient Follow-Up Agent

A read-only clinical decision-support prototype. Outputs are for qualified care-team review and are not diagnoses, prescriptions, or emergency advice.

## Setup

Install dependencies if needed: `pip install pandas openai`. Set `OPENAI_API_KEY` in your environment; it is never stored in this notebook.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

DATA_PATH = Path('patient_data.csv')
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5')
DISCLAIMER = ('Clinical decision support only. A qualified care team must review '
              'all information before making any care decision.')

## Task 1 — Data Loading & Exploratory Analysis

In [8]:
REQUIRED_COLUMNS = {'patient_id', 'patient_name', 'age', 'gender', 'diagnosis',
    'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date',
    'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment',
    'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate',
    'vitals_spo2', 'notes'}
df = pd.read_csv(DATA_PATH)
for column in ['last_visit_date', 'next_scheduled_visit']:
    df[column] = pd.to_datetime(df[column], errors='coerce')

quality = {
    'rows': len(df), 'columns': len(df.columns),
    'missing_required_columns': sorted(REQUIRED_COLUMNS - set(df.columns)),
    'missing_values': df.isna().sum()[lambda s: s.gt(0)].to_dict(),
    'duplicate_patient_ids': int(df.patient_id.duplicated().sum()),
    'invalid_dates': {c: int(df[c].isna().sum()) for c in ['last_visit_date', 'next_scheduled_visit']},
    'unexpected_appointment_values': sorted(set(df.missed_last_appointment) - {'Yes', 'No'})
}
profile = {
    'age': df.age.describe().round(1).to_dict(),
    'top_diagnoses': df.diagnosis.value_counts().head(10).to_dict(),
    'top_medications': df.current_medication.value_counts().head(10).to_dict(),
    'missed_appointments': int(df.missed_last_appointment.eq('Yes').sum()),
    'vitals': df[['vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2']].describe().loc[['min', 'mean', 'max']].round(1).to_dict()
}
print('DATA QUALITY:', json.dumps(quality, indent=2, default=str))
print('\nPROFILE:', json.dumps(profile, indent=2, default=str))

DATA QUALITY: {
  "rows": 100,
  "columns": 18,
  "missing_required_columns": [],
  "missing_values": {},
  "duplicate_patient_ids": 0,
  "invalid_dates": {
    "last_visit_date": 0,
    "next_scheduled_visit": 0
  },
  "unexpected_appointment_values": []
}

PROFILE: {
  "age": {
    "count": 100.0,
    "mean": 55.4,
    "std": 15.3,
    "min": 28.0,
    "25%": 44.0,
    "50%": 55.0,
    "75%": 68.0,
    "max": 82.0
  },
  "top_diagnoses": {
    "Heart Failure": 12,
    "Hypothyroidism": 9,
    "Hypertension, Anemia": 8,
    "COPD": 7,
    "Osteoarthritis": 7,
    "Type 2 Diabetes, Hypertension": 7,
    "Depression": 7,
    "Type 2 Diabetes": 6,
    "COPD, Heart Failure": 6,
    "Asthma": 6
  },
  "top_medications": {
    "Furosemide 40mg": 12,
    "Levothyroxine 50mcg": 9,
    "Amlodipine 5mg, Ferrous Sulfate 200mg": 8,
    "Tiotropium 18mcg": 7,
    "Ibuprofen 400mg": 7,
    "Metformin 500mg, Amlodipine 5mg": 7,
    "Sertraline 50mg": 7,
    "Metformin 500mg": 6,
    "Tiotropium 18mc

**Observed dataset facts:** 100 rows, 18 expected columns, no missing values, duplicate patient IDs, or invalid dates; 15 records have a missed last appointment. Diagnosis, medication, lab, and vital values vary across patients, so no universal clinical conclusion should be inferred from this descriptive profile.

## Task 2 — Define Agent Tools

Tools return only dataset fields and errors. They do not diagnose, rank, contact, schedule, or modify records. Names are intentionally omitted from tool responses to minimize exposure of direct identifiers.

In [9]:
DISPLAY_COLUMNS = ['patient_id', 'age', 'gender', 'diagnosis', 'current_medication',
    'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit',
    'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic',
    'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']

def serialize(frame):
    return json.loads(frame[DISPLAY_COLUMNS].to_json(orient='records', date_format='iso'))

def get_patient_record(patient_id):
    result = df.loc[df.patient_id.astype(str).str.upper().eq(str(patient_id).upper())]
    return {'error': 'Unknown patient_id'} if result.empty else {'record': serialize(result)[0]}

def get_patient_summary(patient_id):
    record = get_patient_record(patient_id)
    if 'error' in record:
        return record
    r = record['record']
    keys = ['patient_id', 'diagnosis', 'current_medication', 'lab_test', 'lab_value',
            'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit',
            'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic',
            'vitals_heart_rate', 'vitals_spo2', 'notes']
    return {'summary': {key: r[key] for key in keys}}

def get_risk_context(patient_id):
    record = get_patient_record(patient_id)
    if 'error' in record:
        return record
    r = record['record']
    keys = ['patient_id', 'diagnosis', 'lab_test', 'lab_value', 'lab_unit',
            'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate',
            'vitals_spo2', 'days_since_last_visit', 'missed_last_appointment', 'notes']
    return {'risk_relevant_source_fields': {key: r[key] for key in keys}}

def get_missed_appointment_candidates():
    return {'count': int(df.missed_last_appointment.eq('Yes').sum()),
            'candidates': serialize(df.loc[df.missed_last_appointment.eq('Yes')])}

TOOL_FUNCTIONS = {'get_patient_record': get_patient_record,
                  'get_patient_summary': get_patient_summary,
                  'get_risk_context': get_risk_context,
                  'get_missed_appointment_candidates': lambda: get_missed_appointment_candidates()}

In [5]:
PATIENT_ID_SCHEMA = {'type': 'object', 'properties': {'patient_id': {'type': 'string'}},
                     'required': ['patient_id'], 'additionalProperties': False}
TOOLS = [
    {'type': 'function', 'name': 'get_patient_record', 'description': 'Retrieve source fields for one patient ID.', 'parameters': PATIENT_ID_SCHEMA, 'strict': True},
    {'type': 'function', 'name': 'get_patient_summary', 'description': 'Retrieve a concise source-grounded summary for one patient ID.', 'parameters': PATIENT_ID_SCHEMA, 'strict': True},
    {'type': 'function', 'name': 'get_risk_context', 'description': 'Retrieve risk-relevant source fields, without clinical interpretation, for one patient ID.', 'parameters': PATIENT_ID_SCHEMA, 'strict': True},
    {'type': 'function', 'name': 'get_missed_appointment_candidates', 'description': 'Retrieve source-grounded records for patients with missed last appointments.', 'parameters': {'type': 'object', 'properties': {}, 'additionalProperties': False}, 'strict': True}
]

## Task 3 — Implement the Agentic Loop

In [10]:
SYSTEM_PROMPT = f'''You are MediCare, a read-only clinical-support assistant.
Use tools before any substantive patient-specific answer. Ground every statement in returned fields;
identify missing or uncertain information; do not diagnose, prescribe, or give emergency advice.
For missed appointments, prioritize candidates for human review using only returned evidence, explain
the evidence and uncertainty, and suggest non-directive outreach considerations. Do not contact or
schedule anyone. Use patient IDs, not names. End with: {DISCLAIMER}'''

def run_agent(user_request, client=None):
    if client is None:
        if OpenAI is None:
            return 'OpenAI package is unavailable. Install it, set OPENAI_API_KEY, then rerun.'
        if not os.getenv('OPENAI_API_KEY'):
            return 'OPENAI_API_KEY is not set; live LLM analysis was not run.'
        client = OpenAI()
    messages = [{'role': 'user', 'content': user_request}]
    try:
        for _ in range(6):
            response = client.responses.create(model=MODEL, instructions=SYSTEM_PROMPT, input=messages, tools=TOOLS, store=False)
            calls = [item for item in response.output if item.type == 'function_call']
            if not calls:
                return response.output_text or 'The model returned no text.'
            messages.extend(response.output)
            for call in calls:
                try:
                    result = TOOL_FUNCTIONS[call.name](**json.loads(call.arguments))
                except (KeyError, TypeError, json.JSONDecodeError) as error:
                    result = {'error': f'Tool request could not be completed: {error}'}
                messages.append({'type': 'function_call_output', 'call_id': call.call_id, 'output': json.dumps(result)})
        return 'Tool-call limit reached; no clinical conclusion was generated.'
    except Exception as error:
        return f'LLM API request failed safely: {type(error).__name__}: {error}'

## Task 4 — Single Patient Analysis

With a configured API key, the LLM autonomously selects and calls patient tools before producing its review. The conditional run keeps the notebook executable without exposing credentials.

In [6]:
PATIENT_ID = 'P0001'

summary_result = get_patient_summary(PATIENT_ID)
risk_result = get_risk_context(PATIENT_ID)

if 'error' in summary_result or 'error' in risk_result:
    print(json.dumps({'patient_id': PATIENT_ID, 'error': 'Patient record could not be retrieved.'}, indent=2))
else:
    summary = summary_result['summary']
    risk_context = risk_result['risk_relevant_source_fields']
    required_evidence = {
        'patient_id', 'diagnosis', 'current_medication', 'lab_test', 'lab_value',
        'lab_unit', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate',
        'vitals_spo2', 'last_visit_date', 'next_scheduled_visit',
        'days_since_last_visit', 'missed_last_appointment', 'notes'
    }
    assert required_evidence.issubset(summary)
    assert required_evidence - set(risk_context) == {
        'current_medication', 'last_visit_date', 'next_scheduled_visit'
    }

    print('SOURCE EVIDENCE')
    print(json.dumps(summary, indent=2, default=str))
    print('\nRISK CONTEXT SOURCE FIELDS')
    print(json.dumps(risk_context, indent=2, default=str))
    print('\nAGENT ANALYSIS')
    print(run_agent(
        f'''Analyze patient {PATIENT_ID}. First call get_patient_summary and get_risk_context.
Use only the returned fields to summarize diagnoses, medication, lab result, vitals, visits,
appointment status, and notes. Cite the supporting values for each risk insight, identify missing
or uncertain information, and provide non-directive follow-up considerations for clinician review.
Do not diagnose, prescribe, or invent thresholds. End with the required clinical-support disclaimer.'''
    ))
    print('\nDISCLAIMER')
    print(DISCLAIMER)


SOURCE EVIDENCE
{
  "patient_id": "P0001",
  "diagnosis": "Type 2 Diabetes",
  "current_medication": "Metformin 500mg",
  "lab_test": "HbA1c",
  "lab_value": 9.9,
  "lab_unit": "%",
  "last_visit_date": "2024-04-24T00:00:00.000",
  "next_scheduled_visit": "2024-06-08T00:00:00.000",
  "days_since_last_visit": 251,
  "missed_last_appointment": "No",
  "vitals_bp_systolic": 174,
  "vitals_bp_diastolic": 70,
  "vitals_heart_rate": 95,
  "vitals_spo2": 96,
  "notes": "Patient reports good medication adherence."
}

RISK CONTEXT SOURCE FIELDS
{
  "patient_id": "P0001",
  "diagnosis": "Type 2 Diabetes",
  "lab_test": "HbA1c",
  "lab_value": 9.9,
  "lab_unit": "%",
  "vitals_bp_systolic": 174,
  "vitals_bp_diastolic": 70,
  "vitals_heart_rate": 95,
  "vitals_spo2": 96,
  "days_since_last_visit": 251,
  "missed_last_appointment": "No",
  "notes": "Patient reports good medication adherence."
}

AGENT ANALYSIS
OPENAI_API_KEY is not set; live LLM analysis was not run.

DISCLAIMER
Clinical decision 

## Task 5 — Missed Appointment Follow-Up

The agent retrieves only patients whose `missed_last_appointment` is `Yes`, then produces a reviewable, evidence-based priority list. It never contacts patients or makes scheduling changes.

In [11]:
missed_result = get_missed_appointment_candidates()

if 'error' in missed_result:
    print(json.dumps(missed_result, indent=2))
else:
    candidates = missed_result['candidates']
    assert missed_result['count'] == len(candidates)
    assert all(record['missed_last_appointment'] == 'Yes' for record in candidates)

    print('MISSED-APPOINTMENT SOURCE EVIDENCE')
    print(json.dumps({
        'count': missed_result['count'],
        'patients': candidates
    }, indent=2, default=str))
    print('\nAGENT FOLLOW-UP REVIEW')
    print(run_agent(
        '''Identify patients who missed their last appointment. First call
get_missed_appointment_candidates and use only the returned records. Produce a prioritized,
care-team review list with one item per returned patient. For each item include patient_id,
missed_last_appointment, relevant diagnosis, medication, lab result, vital signs, days since last
visit, notes, outreach urgency, a non-directive suggested action, rationale tied to source fields,
and uncertainty or missing information. Do not diagnose, prescribe, contact patients, or schedule
appointments. Make clear that priorities and actions require human review.'''
    ))
    print('\nHUMAN-REVIEW NOTE')
    print('This is a review list only; the care team must validate each item before any outreach or scheduling action.')
    print('\nDISCLAIMER')
    print(DISCLAIMER)


MISSED-APPOINTMENT SOURCE EVIDENCE
{
  "count": 15,
  "patients": [
    {
      "patient_id": "P0018",
      "age": 36,
      "gender": "Female",
      "diagnosis": "Heart Failure",
      "current_medication": "Furosemide 40mg",
      "lab_test": "BNP",
      "lab_value": 952.3,
      "lab_unit": "pg/mL",
      "last_visit_date": "2024-07-27T00:00:00.000",
      "next_scheduled_visit": "2024-09-10T00:00:00.000",
      "days_since_last_visit": 157,
      "missed_last_appointment": "Yes",
      "vitals_bp_systolic": 160,
      "vitals_bp_diastolic": 87,
      "vitals_heart_rate": 85,
      "vitals_spo2": 96,
      "notes": "Patient education provided on diet and lifestyle."
    },
    {
      "patient_id": "P0031",
      "age": 68,
      "gender": "Female",
      "diagnosis": "Chronic Kidney Disease",
      "current_medication": "Losartan 50mg",
      "lab_test": "eGFR",
      "lab_value": 52.1,
      "lab_unit": "mL/min/1.73m\u00b2",
      "last_visit_date": "2024-08-06T00:00:00.000",
 